In [ ]:
import duckdb
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
db_path = "/content/drive/MyDrive/RDV_Project/processed_data/rdv_project.duckdb"

con = duckdb.connect(db_path)

print("Connected")

Connected


In [ ]:
import duckdb

db_path = "/content/drive/MyDrive/RDV_Project/processed_data/rdv_project.duckdb"

con = duckdb.connect(db_path)

print("Connected")

Connected


In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name


In [ ]:
import pandas as pd
import duckdb

file_path = "/content/drive/MyDrive/RDV_Project/processed_data/taxi_weather.parquet"

df = pd.read_parquet(file_path)

print(df.shape)

(22012724, 16)


In [ ]:
db_path = "/content/drive/MyDrive/RDV_Project/processed_data/rdv_project.duckdb"

con = duckdb.connect(db_path)

print("Connected")

Connected


In [ ]:
fact_taxi_trip = df[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pickup_date",
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "trip_duration",
        "trip_category",
        "time_category"
    ]
]

In [ ]:
con.register(
    "fact_taxi_trip_df",
    fact_taxi_trip
)

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fact_taxi_trip AS
SELECT * FROM fact_taxi_trip_df
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,fact_taxi_trip
1,fact_taxi_trip_df


In [ ]:
trip_by_hour = con.execute("""
SELECT
    pickup_hour,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY pickup_hour
ORDER BY pickup_hour
""").fetchdf()

trip_by_hour.head()

,pickup_hour,total_trip
0,0,686158
1,1,451034
2,2,295243
3,3,199647
4,4,153601


In [ ]:
fact_weather = df[
    [
        "pickup_date",
        "avg_temperature",
        "precipitation",
        "rain",
        "weather_condition"
    ]
].drop_duplicates()

In [ ]:
con.register(
    "fact_weather_df",
    fact_weather
)

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fact_weather AS
SELECT * FROM fact_weather_df
""")

In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,fact_taxi_trip
1,fact_taxi_trip_df
2,fact_weather
3,fact_weather_df


In [ ]:
trip_by_hour = con.execute("""
SELECT
    pickup_hour,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY pickup_hour
ORDER BY pickup_hour
""").fetchdf()

trip_by_hour.head()

,pickup_hour,total_trip
0,0,686158
1,1,451034
2,2,295243
3,3,199647
4,4,153601


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE agg_trip_by_hour AS
SELECT
    pickup_hour,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY pickup_hour
ORDER BY pickup_hour
""")

In [ ]:
trip_characteristics = con.execute("""
SELECT
    time_category,
    AVG(trip_distance) AS avg_distance,
    AVG(trip_duration) AS avg_duration,
    AVG(fare_amount) AS avg_fare
FROM fact_taxi_trip
GROUP BY time_category
""").fetchdf()

trip_characteristics

,time_category,avg_distance,avg_duration,avg_fare
0,Malam,5.747828,14.771308,18.908821
1,Siang,5.590490,18.073854,20.114269
2,Pagi,9.437325,16.330253,19.294306


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE agg_trip_characteristics AS
SELECT
    time_category,
    AVG(trip_distance) AS avg_distance,
    AVG(trip_duration) AS avg_duration,
    AVG(fare_amount) AS avg_fare
FROM fact_taxi_trip
GROUP BY time_category
""")

In [ ]:
top_pickup_zone = con.execute("""
SELECT
    PULocationID,
    COUNT(*) AS total_pickup
FROM fact_taxi_trip
GROUP BY PULocationID
ORDER BY total_pickup DESC
LIMIT 20
""").fetchdf()

top_pickup_zone

,PULocationID,total_pickup
0,237,1007100
1,161,993775
2,236,899964
3,132,890747
4,230,719878
5,186,709360
6,162,703996
7,142,655559
8,138,605238
9,234,603551


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE agg_top_pickup_zone AS
SELECT
    PULocationID,
    COUNT(*) AS total_pickup
FROM fact_taxi_trip
GROUP BY PULocationID
ORDER BY total_pickup DESC
""")

In [ ]:
travel_pattern = con.execute("""
SELECT
    time_category,
    PULocationID,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY time_category, PULocationID
ORDER BY total_trip DESC
""").fetchdf()

travel_pattern.head()

,time_category,PULocationID,total_trip
0,Siang,237,441973
1,Malam,161,431186
2,Malam,132,421740
3,Siang,161,405114
4,Siang,236,402525


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE agg_travel_pattern AS
SELECT
    time_category,
    PULocationID,
    COUNT(*) AS total_trip
FROM fact_taxi_trip
GROUP BY time_category, PULocationID
""")

In [ ]:
weather_effect = con.execute("""
SELECT
    fw.weather_condition,
    COUNT(*) AS total_trip,
    AVG(ft.trip_distance) AS avg_distance,
    AVG(ft.trip_duration) AS avg_duration
FROM fact_taxi_trip ft
JOIN fact_weather fw
ON ft.pickup_date = fw.pickup_date
GROUP BY fw.weather_condition
""").fetchdf()

weather_effect

,weather_condition,total_trip,avg_distance,avg_duration
0,Hujan Ringan,5972647,6.657216,16.635996
1,Tidak Hujan,11671863,6.413749,15.886162
2,Hujan Lebat,4368190,6.568458,16.736980
3,None,24,3.960417,17.536111


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE agg_weather_effect AS
SELECT
    fw.weather_condition,
    COUNT(*) AS total_trip,
    AVG(ft.trip_distance) AS avg_distance,
    AVG(ft.trip_duration) AS avg_duration
FROM fact_taxi_trip ft
JOIN fact_weather fw
ON ft.pickup_date = fw.pickup_date
GROUP BY fw.weather_condition
""")

In [ ]:
con.execute("""
SHOW TABLES
""").fetchdf()

,name
0,agg_top_pickup_zone
1,agg_travel_pattern
2,agg_trip_by_hour
3,agg_trip_characteristics
4,agg_weather_effect
5,fact_taxi_trip
6,fact_taxi_trip_df
7,fact_weather
8,fact_weather_df


In [ ]:
output_path = "/content/drive/MyDrive/RDV_Project/processed_data/"

In [ ]:
trip_by_hour.to_parquet(
    output_path + "agg_trip_by_hour.parquet",
    index=False
)

trip_characteristics.to_parquet(
    output_path + "agg_trip_characteristics.parquet",
    index=False
)

top_pickup_zone.to_parquet(
    output_path + "agg_top_pickup_zone.parquet",
    index=False
)

travel_pattern.to_parquet(
    output_path + "agg_travel_pattern.parquet",
    index=False
)

weather_effect.to_parquet(
    output_path + "agg_weather_effect.parquet",
    index=False
)

print("All aggregate data saved")

All aggregate data saved


In [ ]:
con.close()